# Document Loaders

- LangChain은 다양한 파일 형식을 통일된 `Document` 객체로 변환하는 로더를 제공함
- `Document`는 `page_content`(본문 텍스트)와 `metadata`(딕셔너리)로 구성되며, 이후 분할 및 임베딩 단계에서 일관된 인터페이스로 사용됨.

### 참고자료
- https://docs.langchain.com/oss/python/integrations/document_loaders
- https://wikidocs.net/231429

## 1. 환경 준비

## (1) 라이브러리 설치

처음 실행하는 환경이라면 아래 셀의 주석을 해제하고 실행합니다. 이미 `pyproject.toml` 또는 `requirements.txt`로 설치했다면 실행하지 않아도 됩니다.


In [ ]:
# 필요한 라이브러리 설치
# uv add langchain-openai numpy scikit-learn

## (2) API Key 설정

API 키는 코드에 직접 작성하지 않습니다. 권장 방식은 `.env` 파일에 저장하는 것입니다.

```text
# OpenAI를 사용할 때
OPENAI_API_KEY=sk-...

# Gemini를 사용할 때
GOOGLE_API_KEY=...

```


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

print("OPENAI_API_KEY:", "있음" if os.getenv("OPENAI_API_KEY") else "없음")
print("GOOGLE_API_KEY:", "있음" if os.getenv("GOOGLE_API_KEY") else "없음")

OPENAI_API_KEY: 있음
GOOGLE_API_KEY: 있음


## 2. Document Loaders

### 2.1 TextLoader
- 단일 텍스트 파일

In [5]:
# uv add langchain_community
from langchain_community.document_loaders import TextLoader
from pathlib import Path

hr_policy_path = Path('data2/hr_policy.txt')

In [6]:
# encoding='utf-8' : 한글이 깨지지 않도록.
loader = TextLoader(str(hr_policy_path), encoding='utf=8')

# 파일을 읽어서 랭체인 Document 객체 리스트로 변환
documents = loader.load()

In [8]:
print(f"로드된 문서 수: {len(documents)}")
print(f"문서 길이: {len(documents[0].page_content)} 글자")
print(f"메타데이터: {documents[0].metadata}")
print(f"\n미리보기 (처음 300자):")
print(documents[0].page_content[:300])

로드된 문서 수: 1
문서 길이: 3330 글자
메타데이터: {'source': 'data2\\hr_policy.txt'}

미리보기 (처음 300자):
# [인사 정책 매뉴얼] 즐겁고 공정한 직장 문화를 위한 가이드

본 안내서는 우리 회사의 핵심 인사 원칙과 규정을 담고 있습니다. 모든 임직원은 본 정책을 준수하며, 상호 존중과 성과 중심의 문화를 함께 만들어 갑니다.

---

## 1. 휴가 및 근태 정책

우리 회사는 임직원의 충분한 휴식과 일과 삶의 균형(Work-Life Balance)을 전폭적으로 지원합니다.

### 1.1 연차 유급 휴가

* **발생 기준:** * **신입사원:** 입사 1년 미만 시, 1개월 개근 시 1일씩 발생 (최대 11일).
* **정기 연


### 2.2 DirectoryLoader
- 폴더 전체 로드 (다중 포맷)

In [9]:
from langchain_community.document_loaders import DirectoryLoader

In [11]:
company_docs_path = Path("data2/company_docs")
print(company_docs_path.resolve())
print(company_docs_path.exists())


C:\Users\playdata2\work_space(playdata)\SKN30_playdata\LLM\data2\company_docs
False


In [ ]:
def load_all_documents(data_dir: Path):
    """폴더에서 .txt와 .md 파일을 모두 로드합니다."""
    all_docs = []
    for pattern in ["**/*.txt", "**/*.md"]:
        loader = DirectoryLoader(
            str(data_dir),
            glob=pattern,
            loader_cls=TextLoader,
            loader_kwargs={"encoding": "utf-8"},
        )
        all_docs.extend(loader.load())
    return all_docs

In [ ]:
all_docs = load_all_documents(company_docs_path)
print(f"로드된 문서 수: {len(all_docs)}")
for doc in all_docs[:5]:
    print(f"   - {doc.metadata['source']}: {len(doc.page_content)} 글자")

### 2.3. PyPDF Loader
- PDF 파일 로드
- 이미지+텍스트 페이지 내 텍스트 추출

In [12]:
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
filename = Path('data2/[이슈리포트 2022-2호] 혁신성장 정책금융 동향.pdf')
print(company_docs_path.resolve())

- OCR 기능 활용하여 이미지-텍스트 혼합 페이지 내 텍스트 추출하기

In [ ]:
# #OCR기능 위해 설치
# uv add rapidocr

- 아래 셀은 실행 시 많은 시간이 소요됩니다.

- 페이지 내 테이블 추출하기

### 2.4. PyPDFium2

- 이미지+텍스트 페이지 내 텍스트 추출

In [ ]:
# #PyPDFium2 설치
# uv add pypdfium2

- 페이지 내 테이블 추출하기

### 2.5. PyPDFLoader vs PyPDFium2Loader

- PyPDFLoader의 텍스트 추출 소요 시간

In [ ]:
%%time

loader = PyPDFLoader(filename, mode="page")

pages = loader.load()

- PyPDFium2의 텍스트 추출 소요 시간

In [ ]:
%%time

loader = PyPDFium2Loader(filename)

pages = loader.load()